In [ ]:
# INIT BORNEO : scraping complet de l'historique et création de la base initiale

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
import pandas as pd
import time
import os

# Configuration
EMAIL = os.getenv("BORNEO_EMAIL", "your_email_here")
PASSWORD = os.getenv("BORNEO_PASSWORD", "your_password_here")
URL_LOGIN = "https://restaurant.borneoapp.com"
HISTORY_URL = "https://restaurant.borneoapp.com/multiview/store/ChickenStreetPontaultCombault/history"
FICHIER_CSV_INIT = "BDD_Client_Borneo.csv"
WEBDRIVER_PATH = os.getenv("BORNEO_DRIVER_PATH", r"C:\path\to\msedgedriver.exe")

service = Service(WEBDRIVER_PATH)
driver = webdriver.Edge(service=service)
client_data = []


def get_text_or_default(wait, xpath, default="Non trouvé"):
    try:
        return wait.until(EC.presence_of_element_located((By.XPATH, xpath))).text.strip()
    except:
        return default


try:
    driver.get(URL_LOGIN)
    wait = WebDriverWait(driver, 30)

    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Se connecter')]"))).click()
    wait.until(EC.presence_of_element_located((By.ID, "login-email-input"))).send_keys(EMAIL)
    driver.find_element(By.ID, "login-password-input").send_keys(PASSWORD)
    driver.find_element(By.XPATH, "//button[contains(text(), 'Se connecter')]").click()

    time.sleep(3)
    driver.get(HISTORY_URL)

    dropdown = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "timeslot-selector")))
    dropdown.click()
    time.sleep(1)

    all_options = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, "//optgroup[@label='Mois']/option"))
    )

    for option in all_options:
        month_text = option.text
        print(f"Traitement du mois : {month_text}")
        option.click()
        time.sleep(2)

        commandes = driver.find_elements(By.XPATH, "//unor-order-report-item")
        if not commandes:
            print(f"Aucune commande détectée pour {month_text}")
            continue

        for i, commande in enumerate(commandes):
            try:
                print(f"Commande {i + 1}/{len(commandes)}")
                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(commande)).click()
                WebDriverWait(driver, 20).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[@class='btn d-block w-100 py-2']"))
                ).click()

                prenom = get_text_or_default(wait, "//dt[text()='Prénom']/following-sibling::dd")
                phone = get_text_or_default(wait, "//a[contains(@href, 'tel:')]")
                email = get_text_or_default(wait, "//dt[text()='Email']/following-sibling::dd")
                nb_commandes = get_text_or_default(wait, "//span[@class='text-info ng-star-inserted']")

                client_data.append({
                    "Mois": month_text,
                    "Prénom": prenom,
                    "Téléphone": phone,
                    "Email": email,
                    "Nombre de commandes": nb_commandes,
                    "Restaurant": "Chicken Street Pontault Combault",
                    "Plateforme": "Borneo"
                })

                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(commande)).click()
                time.sleep(1)

            except Exception as e:
                print(f"Erreur commande {i + 1} : {e}")

finally:
    driver.quit()

# Enregistrement de la base initiale
df_init = pd.DataFrame(client_data)
df_init.to_csv(FICHIER_CSV_INIT, index=False, encoding="utf-8")
print(f"\nBase initiale enregistrée dans : {FICHIER_CSV_INIT} ({len(df_init)} lignes)")

In [ ]:
# SCRAPING RÉCENT BORNEO : scraper uniquement les 28 derniers jours

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
import pandas as pd
import time
import os

# Configuration
EMAIL = os.getenv("BORNEO_EMAIL", "your_email_here")
PASSWORD = os.getenv("BORNEO_PASSWORD", "your_password_here")
URL_LOGIN = "https://restaurant.borneoapp.com"
HISTORY_URL = "https://restaurant.borneoapp.com/multiview/store/ChickenStreetPontaultCombault/history"
FICHIER_CSV_TEMP = "Borneo_Temp.csv"
WEBDRIVER_PATH = os.getenv("BORNEO_DRIVER_PATH", r"C:\path\to\msedgedriver.exe")

service = Service(WEBDRIVER_PATH)
driver = webdriver.Edge(service=service)
client_data = []


def get_text_or_default(wait, xpath, default="Non trouvé"):
    try:
        return wait.until(EC.presence_of_element_located((By.XPATH, xpath))).text.strip()
    except:
        return default


try:
    driver.get(URL_LOGIN)
    wait = WebDriverWait(driver, 20)

    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Se connecter')]"))).click()
    wait.until(EC.presence_of_element_located((By.ID, "login-email-input"))).send_keys(EMAIL)
    driver.find_element(By.ID, "login-password-input").send_keys(PASSWORD)
    driver.find_element(By.XPATH, "//button[contains(text(), 'Se connecter')]").click()

    time.sleep(3)
    driver.get(HISTORY_URL)

    dropdown = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "timeslot-selector")))
    dropdown.click()
    time.sleep(1)

    option_28j = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//option[contains(text(), '28 derniers jours')]"))
    )
    option_28j.click()
    time.sleep(2)

    commandes = driver.find_elements(By.XPATH, "//unor-order-report-item")

    if not commandes:
        print("Aucune commande détectée pour les 28 derniers jours")
    else:
        for i, commande in enumerate(commandes):
            try:
                print(f"Commande {i + 1}/{len(commandes)}")
                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(commande)).click()
                WebDriverWait(driver, 20).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[@class='btn d-block w-100 py-2']"))
                ).click()

                prenom = get_text_or_default(wait, "//dt[text()='Prénom']/following-sibling::dd")
                phone = get_text_or_default(wait, "//a[contains(@href, 'tel:')]")
                email = get_text_or_default(wait, "//dt[text()='Email']/following-sibling::dd")
                nb_commandes = get_text_or_default(wait, "//span[@class='text-info ng-star-inserted']")

                client_data.append({
                    "Mois": "28 derniers jours",
                    "Prénom": prenom,
                    "Téléphone": phone,
                    "Email": email,
                    "Nombre de commandes": nb_commandes,
                    "Restaurant": "Chicken Street Pontault Combault",
                    "Plateforme": "Borneo"
                })

                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(commande)).click()
                time.sleep(1)

            except Exception as e:
                print(f"Erreur commande {i + 1} : {e}")

finally:
    driver.quit()

# Enregistrement des données récentes
df_temp = pd.DataFrame(client_data)
df_temp.to_csv(FICHIER_CSV_TEMP, index=False, encoding="utf-8")
print(f"\nDonnées des 28 derniers jours enregistrées dans : {FICHIER_CSV_TEMP} ({len(df_temp)} lignes)")

In [ ]:
import pandas as pd

FICHIER_CSV_BASE = "BDD_Client_Borneo.csv"
FICHIER_CSV_TEMP = "Borneo_Temp.csv"

# Chargement
df_old = pd.read_csv(FICHIER_CSV_BASE)
df_new = pd.read_csv(FICHIER_CSV_TEMP)

# Alignement des colonnes
for col in df_old.columns:
    if col not in df_new.columns:
        df_new[col] = None

df_new = df_new[df_old.columns]

# Fusion
df_combined = pd.concat([df_old, df_new], ignore_index=True)

# Renommage intelligent
renommages = {
    "telephone": "Téléphone",
    "numéro": "Téléphone",
    "Telephone": "Téléphone",
    "email": "Email",
    "restaurant": "Restaurant",
    "prenom": "Prénom",
    "nom": "Nom"
}

df_combined.rename(
    columns={k: v for k, v in renommages.items() if k in df_combined.columns},
    inplace=True
)

# Vérification des colonnes pour scoring
colonnes_infos = [col for col in ["Téléphone", "Email", "Restaurant", "Prénom"] if col in df_combined.columns]

if colonnes_infos:
    df_combined["infos"] = df_combined[colonnes_infos].notnull().sum(axis=1)
    df_combined = df_combined.sort_values(by="infos", ascending=False)

    if "Téléphone" in df_combined.columns:
        df_combined = df_combined.drop_duplicates(subset="Téléphone", keep="first")

    df_combined.drop(columns="infos", inplace=True)
else:
    print("Pas de colonnes clés pour nettoyage — aucune déduplication appliquée.")

# Sauvegarde
df_combined.to_csv(FICHIER_CSV_BASE, index=False, encoding="utf-8")
print("Fusion Borneo terminée sans erreur.")